# Lesson 7 | The first RTL neuron

Register-Transfer Level (RTL) describes calculations and updates between registers. Implement an Integrate-and-Fire (IF) rule: accumulate input, compare threshold, reset on firing. The circuit contains an adder, comparator, mux, and two state registers.


## 1. Concept ledger

**Known:** register, clock edge, RTL module/port, and one simple `always_ff` example.

**New:** combinational path, `always_comb`, and how it works with sequential `always_ff`.

**Preview:** testbench and waveform wait until next lesson.


## 2. Rules for each clock edge

1. Add old `membrane_v` and input to obtain `candidate`.
2. Compare `candidate >= threshold` for `spike_next`.
3. Select `reset_value` on firing, otherwise `candidate`, as `next_v`.
4. At a positive edge, active reset stores `reset_value` and clears `spike`; otherwise store `next_v` and `spike_next`.

Signals are signed eight-bit values. Main examples stay within −128 to 127; a separate experiment shows low-eight-bit overflow. This IF example implements integration, threshold, and reset; leak requires an additional calculation.


## 3. Replay the rule with a Python oracle

Predict spike cycles for threshold 4 and inputs `[1,1,1,1,2,2]`, then run.


In [ ]:
def tutorial_if_step(state, input_current, threshold, reset_value=0):
    candidate = state + input_current
    spike = candidate >= threshold
    next_state = reset_value if spike else candidate
    return next_state, spike, candidate

state = 0
for cycle, current in enumerate([1, 1, 1, 1, 2, 2]):
    next_state, spike, candidate = tutorial_if_step(state, current, 4)
    print(f'cycle={cycle}: state={state}, input={current}, candidate={candidate}, spike={spike}, next={next_state}')
    state = next_state


## 4. Structure

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 300" role="img" aria-label="l07-flow" style="width:100%;height:auto">
<defs><marker id="l07-flow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs>
<rect x="20" y="100" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="95.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">membrane_v</text>
<rect x="225" y="100" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="300.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">adder</text>
<rect x="430" y="100" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="505.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">comparator / mux</text>
<rect x="635" y="100" width="180" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="725.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">state + spike regs</text>
<rect x="225" y="15" width="150" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="300.0" y="42.5" text-anchor="middle" font-family="sans-serif" font-size="15">input_current</text>
<rect x="430" y="15" width="150" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="505.0" y="42.5" text-anchor="middle" font-family="sans-serif" font-size="15">threshold / reset</text>
<rect x="635" y="215" width="180" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="725.0" y="242.5" text-anchor="middle" font-family="sans-serif" font-size="15">clk / rst_n</text>
<path d="M170 130 H225" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M375 130 H430" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<text x="402" y="118" text-anchor="middle" font-family="sans-serif" font-size="14">candidate</text>
<path d="M580 130 H635" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<text x="607" y="185" text-anchor="middle" font-family="sans-serif" font-size="14">next_v / spike_next</text>
<path d="M300 60 V100" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M505 60 V100" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M725 215 V160" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M815 130 H830 V285 H95 V160" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<text x="390" y="278" text-anchor="middle" font-family="sans-serif" font-size="14">stored membrane_v</text>
</svg>


## 5. How `always_comb` and `always_ff` cooperate

Both are standard SystemVerilog keywords. `always_comb` derives sensitivity from relevant reads and executes once at simulation startup. Here it reads state, input, threshold, and reset value to calculate `candidate`, `spike_next`, and `next_v`.

Blocking `=` assignments allow each later statement to read the preceding intermediate result. The comparison sees the candidate just calculated. Both branches assign `next_v`, so it need not retain history. Omitting a branch can describe latch behavior; automatic sensitivity cannot complete the assignment.

`always_ff @(posedge clk)` uses `<=` to store results in `membrane_v` and `spike`. Right sides are evaluated before those register updates. New state then causes the combinational logic to recalculate, so the candidate before an edge may differ from the candidate after it.


## 6. Complete teaching RTL

```systemverilog
module tutorial_if_neuron (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_current,
    input  logic signed [7:0] threshold,
    input  logic signed [7:0] reset_value,
    output logic signed [7:0] membrane_v,
    output logic              spike
);
    logic signed [7:0] candidate;
    logic signed [7:0] next_v;
    logic spike_next;

    always_comb begin
        candidate = membrane_v + input_current;
        spike_next = candidate >= threshold;

        if (spike_next)
            next_v = reset_value;
        else
            next_v = candidate;
    end

    always_ff @(posedge clk) begin
        if (!rst_n) begin
            membrane_v <= reset_value;
            spike <= 1'b0;
        end else begin
            membrane_v <= next_v;
            spike <= spike_next;
        end
    end
endmodule
```


## 7. Read the circuit from statements

`candidate` and `next_v` are signed eight-bit intermediate signals; `spike_next` is one bit. Addition describes the adder, `>=` the comparator, and `if/else` the mux.

`membrane_v` and `spike` are clocked output state. `1'b0` is a one-bit binary zero; `8'sd4` is an eight-bit signed decimal 4.

Before an edge with state 3, input 1, threshold 4, and reset value 0, the candidate is 4, spike-next is 1, and next state is 0. After that edge, stored state is 0 and spike is 1. With input still 1, combinational signals recalculate to candidate 1, spike-next 0, and next state 1. Stored spike remains 1 until the next positive edge.


### Optional experiment: how can 8-bit overflow break intuition?

The teaching RTL stores `candidate` in an 8-bit signed value with range `-128..127`. If the mathematical result exceeds that range, hardware keeps only the finite-width bit pattern; interpreting it as two's complement can turn the result negative.

Predict what `100 + 50` becomes as an 8-bit signed value, then run the Python experiment below.

In [ ]:
def wrap_signed_8(x: int) -> int:
    return ((x + 128) % 256) - 128

mathematical_sum = 100 + 50
wrapped_sum = wrap_signed_8(mathematical_sum)
print('mathematical sum =', mathematical_sum)
print('8-bit signed result =', wrapped_sum)


The result is `150 → −106`: the low eight bits `10010110` represent −106 in signed two's complement. A positive threshold may therefore suppress firing. Representing 150 requires wider arithmetic and storage; clamping requires explicit saturation.

If reset value is at or above threshold and input is zero, the reset state already satisfies the comparison, so this example fires every subsequent cycle. Formal overflow and parameter restrictions belong to the numerical specification referenced in Project Trace.


## 8. Try It: work one edge by hand

Before an edge: `membrane_v=3, input_current=1, threshold=4, reset_value=0`. Write candidate / spike_next / next_v, then post-edge membrane_v / spike.


## Exercise

[Lesson 7 exercise: separate the combinational path from the register edge](../../exercises/en/07_first_rtl_neuron.ipynb)

The standalone workbook checks this lesson's semantic understanding; the real RTL/testbench experiment in the lesson remains part of the course.

## 9. AI Task

Ask AI to map each of the five contract rules to the exact RTL statements. If it finds a mismatch, report it rather than changing the contract.


## 10. Human Check

Without AI, identify the combinational path and stored state, explain `always_comb` vs `always_ff`, explain how `candidate=4` and post-edge `membrane_v=0` can both be correct, and explain why this teaching module is not formal LIF RTL.


## 11. Engineering Handoff

`rtl/learning/tutorial_if_neuron.sv` demonstrates mapping a known contract into RTL structure. Formal `rtl/neuron/lif_neuron_engine.sv` still waits for v0 semantics and the fixed-point contract.


## 12. Project Trace

- Lesson: `LSN-007`
- Mapping: `RMD-004` teaching precursor
- Formal `MOD-003`: intentionally not created


## 13. Exit Ticket

You can derive a combinational-path-plus-register structure from the contract and identify both parts in RTL. Next lesson adds no neuron behavior; it verifies this one.
